# SDC using a Zero-to-Nodes or a Node-to-Node formulation 

📜 _If you already know the [original paper of Dutt, Greengard & Rokhlin (2000)](https://link.springer.com/content/pdf/10.1023/A:1022338906936.pdf), you may notice that their description of SDC is very different from the one given [in our basic tutorial](../basics/2_sdc.ipynb) ..._

Indeed, we introduced SDC using a **Zero-to-Nodes formulation (Z2N)**, which describes the sweep update from the initial step solution (zero) to a given node. 
Introduced by [Huang, Jia & Minion (2006)](https://doi.org/10.1016/j.jcp.2005.10.004), the Z2N description of SDC aligns with the description of Runge-Kutta methods using Butcher tables in the literature.
Furthermore, it was also used to describe _iterated Runge-Kutta methods_ by [van der Houwen & Sommeijer (1991)](https://doi.org/10.1137/0912054), which can be seen as ancestors of SDC.

However, the original SDC authors used a different formulation, inspired from the _deferred correction_ methods introduced by the pioneering work of [Fox (1947)](https://doi.org/10.1098/rspa.1947.0060) and [Pereyra (1968)](https://link.springer.com/article/10.1007/BF02165307).
This approach, that we denote by **Node-to-Node formulation (N2N)**, describes the sweep update (or correction formula) from one node to the next.
While both formulations can produce identical SDC schemes, they have some fundamental differences from an implementation perspective, and leads to different generalizations of SDC.

## Deriving the N2N formulation

Let's start from the Z2N sweep update on the Dahlquist problem :

$$
{\bf u}^{k+1} - \lambda\Delta{t}Q_\Delta u^{k+1} = {\bf u}_n + \lambda\Delta{t}(Q-Q_\Delta){\bf u}^k,
$$

where ${\bf u} = [u_n, \dots, u_n]^T$ is the initial solution for the time-step, ${\bf u}^k = [\dots, u_m^k, \dots]^T$ the node solution vector and $u_{m}^k \simeq u(t_n+\tau_m\Delta{t}),\; m \in \{1,\dots,M\}$ is the SDC solution at a given quadrature node.
Writing the sweep update for each node individually yield :

$$
u^{k+1}_{m} - \lambda\Delta{t}\sum_{j=1}^{m}q^\Delta_{m,j} u^{k+1}_{j} 
    = u_n 
    + \lambda\Delta{t}\sum_{j=1}^{M}q_{m,j}u^{k}_{j}
    - \lambda\Delta{t}\sum_{j=1}^{m}q^\Delta_{m,j}u^{k}_{j},
$$

where we note $(q^\Delta)_{i,j} := Q_\Delta$ and $(q)_{i,j} := Q$.
Rearranging and regrouping terms, we can write it like this :

$$
u^{k+1}_{m} =
    u_n 
    + \lambda\Delta{t}\sum_{j=1}^{m}q^\Delta_{m,j} (u^{k+1}_{j} - u^{k}_{j}) 
    + \lambda\Delta{t}\sum_{j=1}^{M}q_{m,j}u^{k}_{j}.
$$

Now subtracting the update formula for $u^{k+1}_m$ from $u^{k+1}_{m+1}$,
we get for $m > 0$ (starting from the second node) :

$$
u^{k+1}_{m+1} = u^{k+1}_m
    + \lambda\Delta{t}\sum_{j=1}^{m+1}\left[q^\Delta_{m+1,j} - q^\Delta_{m,j}\right]\left(u^{k+1}_{j} - u^{k}_{j}\right)
    + \lambda\Delta{t}\sum_{j=1}^{M}\left[q_{m+1,j}-q_{m,j}\right]u^{k}_{j},
$$

and for the first node :

$$
u^{k+1}_{1} = u_n 
    + \lambda\Delta{t}q^\Delta_{1,1} (u^{k+1}_{1} - u^{k}_{1}) 
    + \lambda\Delta{t}\sum_{j=1}^{M}q_{1,j}u^{k}_{j}
$$

We call this the **Node-to-node (N2N) sweep update**, since the correction for $u^{k+1}_{m+1}$ explicitly depends on the previous node solution $u^{k+1}_{m}$ (or the initial step solution $u_n$ for the first node).

> 🔔 Note that $q^\Delta_{m,m+1}=0$ because of the lower triangular nature of $Q_\Delta$, which is used in the above formula.

Defining $s_{m+1,j} = q_{m+1,j}-q_{m,j} \; \forall m \in \{1, M-1\}$ and $s_{1,j} = q_{1,j}$,
we note $S$ the matrix built with the $(s)_{i,j}$ coefficients,
and write the **generic N2N sweep update** into matrix formulation :

$$
\begin{pmatrix}
1 \\
-1 & 1 \\
& \ddots & \ddots \\
& & -1 & 1
\end{pmatrix}
({\bf u}^{k+1} - {\bf u}^{k}) 
= \begin{pmatrix}
{\bf u}_n \\ 0 \\ \vdots \\ 0 
\end{pmatrix}
+ \lambda\Delta{t}S_\Delta ({\bf u}^{k+1} - {\bf u}^{k}) + \lambda\Delta{t}S {\bf u}^k,
$$

where $S_\Delta$ is built from the $Q_\Delta$ matrix the same way as $S$ from $Q$. 
Now you may remark that the N2N formula is actually more complex than the Z2N formula given above.

💡 _Thing is : some specific_ $Q_\Delta$ _coefficients allow to simplify a lot this formula ..._

## Backward-Euler sweep

Consider now one of the original SDC form proposed by Dutt, Greengard & Rokhlin : the Backward-Euler sweep.
The associated coefficients are implemented in `qmat`, and considering a simple node distribution we obtain the following $Q_\Delta$ :

In [1]:
from qmat import genQDeltaCoeffs
genQDeltaCoeffs("BE", nodes=[0.1, 0.3, 0.7, 1.0])

array([[0.1, 0. , 0. , 0. ],
       [0.1, 0.2, 0. , 0. ],
       [0.1, 0.2, 0.4, 0. ],
       [0.1, 0.2, 0.4, 0.3]])

This is a triangular matrix with non-zero diagonal (implicit sweep), but with **all non-zero coefficients in each columns being identical**. 
It implies that for $m \in \{1, M-1\}$ :

$$
\lambda\Delta{t}\sum_{j=1}^{m} s^\Delta_{m+1,j}\left(u^{k+1}_{j} - u^{k}_{j}\right) = 
\lambda\Delta{t}\sum_{j=1}^{m}\left(q^\Delta_{m+1,j} - q^\Delta_{m,j}\right)\left(u^{k+1}_{j} - u^{k}_{j}\right) = 0
$$

thus simplifies the N2N formula since only $s^\Delta_{m+1,m+1}$ remains :

$$
u^{k+1}_{m+1} = u^{k+1}_m
    + \lambda\Delta{t}s^\Delta_{m+1,m+1}\left(u^{k+1}_{m+1} - u^{k}_{m+1}\right)
    + \lambda\Delta{t}\sum_{j=1}^{M}s_{m+1,j}u^{k}_{j},
$$

We note $\forall k\;u^{k}_0 := u_n$, such that the formula can be applied on all nodes.
In fact, the $S_\Delta$ matrix is diagonal, as we can see using `qmat` setting `form="N2N"` for the 
`genQDeltaCoeffs` function :

In [2]:
genQDeltaCoeffs("BE", form="N2N", nodes=[0.1, 0.3, 0.7, 1.0])

array([[0.1, 0. , 0. , 0. ],
       [0. , 0.2, 0. , 0. ],
       [0. , 0. , 0.4, 0. ],
       [0. , 0. , 0. , 0.3]])

In this case in particular, we have $s^\Delta_{m+1,m+1} = q^\Delta_{m+1,m+1} = \tau_{m+1}-\tau_{m}$,
where $(\tau_m)_{1\leq m \leq M} \in [0, 1]$ are the nodes positions and $\tau_0=0$.
This is then **exactly** the SDC formula for Backward Euler introduced by
[Dutt, Greengard & Rokhlin](https://link.springer.com/content/pdf/10.1023/A:1022338906936.pdf).

> 🔔 Note that the **correction term** $\lambda\Delta{t}s^\Delta_{m+1,m+1}\left(u^{k+1}_{m+1} - u^{k}_{m+1}\right)$
> depends on the result of the sweep update $u^{k+1}_{m+1}$, requiring then an **implicit** solver. 

## Forward Euler based sweep

An other sweep update was proposed by Dutt, Greengard & Rokhlin using a Forward Euler correction.

Consider the $Q_\Delta$ coefficients implemented in `qmat` for the same node distribution as before :

In [3]:
genQDeltaCoeffs("FE", nodes=[0.1, 0.3, 0.7, 1.0])

array([[0. , 0. , 0. , 0. ],
       [0.2, 0. , 0. , 0. ],
       [0.2, 0.4, 0. , 0. ],
       [0.2, 0.4, 0.3, 0. ]])

Those are similar to the ones obtained for Backward Euler, except they are "shifted" to the left and the diagonal contains only zeros. Looking at the corresponding $S_\Delta$ coefficients, we obtain for :

$$
u^{k+1}_{m+1} = u^{k+1}_m
    + \lambda\Delta{t}s^\Delta_{m,m}\left(u^{k+1}_{m} - u^{k}_{m}\right)
    + \lambda\Delta{t}\sum_{j=1}^{M}s_{m+1,j}u^{k}_{j},
$$

with $s^\Delta_{m,m}=q^\Delta_{m,m}=\tau_{m+1}-\tau_{m}$ where $(\tau_m)_{1\leq m \leq M} \in [0, 1]$ 
are the nodes positions and $\tau_0=0$ (as before),
as we can see looking at the $S_\Delta$ coefficient generated by `qmat` :

In [4]:
genQDeltaCoeffs("FE", form="N2N", nodes=[0.1, 0.3, 0.7, 1.0])

array([[0. , 0. , 0. , 0. ],
       [0.2, 0. , 0. , 0. ],
       [0. , 0.4, 0. , 0. ],
       [0. , 0. , 0.3, 0. ]])

This is similar as for Backward Euler, except the correction term does not depend on $u^{k+1}_{m+1}$ this time, which makes the sweep fully **explicit**.

## Additional notes

Other sweep types have a simplified form in N2N formulation, e.g using the trapeze rule (Crank-Nicholson) : 

In [5]:
genQDeltaCoeffs("TRAP", form="N2N", nodes=[0.1, 0.3, 0.7, 1.0])

array([[0.05, 0.  , 0.  , 0.  ],
       [0.1 , 0.1 , 0.  , 0.  ],
       [0.  , 0.2 , 0.2 , 0.  ],
       [0.  , 0.  , 0.15, 0.15]])

which produce a sweep update of this form :

$$
u^{k+1}_{m+1} = u^{k+1}_m
    + \lambda\Delta{t}s^\Delta_{m+1,m+1}\left(u^{k+1}_{m+1} - u^{k}_{m+1}\right)
    + \lambda\Delta{t}s^\Delta_{m,m}\left(u^{k+1}_{m} - u^{k}_{m}\right)
    + \lambda\Delta{t}\sum_{j=1}^{M}s_{m+1,j}u^{k}_{j}.
$$

From an algorithmic perspective, implementing SDC into N2N form for Backward / Forward Euler or the trapezoid rule
is usually more efficient than the Z2N form, since it requires less floating point operations per sweep 
(correction term use all node solutions in Z2N). 
However, the N2N formulation has two major issues disadvantage when considering generic SDC methods :

1. only a few types of $Q_\Delta$ coefficients allows a simplified N2N formulation, while some other (like LU for instance), don't have a simplified N2N formulation and are implemented more efficiently using the Z2N formulation,
2. the N2N formulation is inherently sequential, and does not allow the design of parallel SDC variants that have a diagonal $Q_\Delta$ matrix; only the Z2N allows parallelism across the nodes.

Furthermore, while the N2N formulation was historically used to describe SDC, many later SDC-related publications
have been using the Z2N formulation since it's more convenient for the analysis, when looking to SDC as a preconditioned
fixed point iteration.